# AI-Powered Data Analysis

In [ ]:
# %pip install langchain-openai==0.3.10 | tail -n 1
# %pip install langchain==0.3.21 | tail -n 1
# %pip install openai==1.68.2 | tail -n 1
# %pip install pandas==2.2.3 | tail -n 1
# %pip install numpy==2.2.4 | tail -n 1
# %pip install scikit-learn==1.6.1 | tail -n 1

In [28]:
import numpy as np
import pandas as pd
import sklearn

import langchain
import openai
import langchain_openai

from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain.agents import create_openai_tools_agent

# Instead of `AgentExecutor` use create_agent from langchain>=V1.0. Check 1.agents-and-tools.ipynb for its implementation
from langchain.agents import AgentExecutor

import glob
import os
from typing import List, Optional, Dict

from dotenv import load_dotenv
load_dotenv()

True

## Tools

## Tool to list csv files in current working directly

In [7]:
@tool
def list_csv_files() -> Optional[List[str]]:
    """List all CSV file names in the local directory.

    Returns:
        A list containing CSV file names.
        If no CSV files are found, returns None.
    """
    csv_files = glob.glob(os.path.join(os.getcwd(), "*.csv")) # glob.glob is used to return list of files that match given pattern
    if not csv_files:
        return None
    return [os.path.basename(file) for file in csv_files]

In [8]:
list_csv_files.invoke({})

['regression-dataset.csv', 'classification-dataset.csv']

In [9]:
print("Tool Name:", list_csv_files.name)
print("Tool Description:", list_csv_files.description)
print("Tool Arguments:", list_csv_files.args)

Tool Name: list_csv_files
Tool Description: List all CSV file names in the local directory.

    Returns:
        A list containing CSV file names.
        If no CSV files are found, returns None.
Tool Arguments: {}


### Tool to load datasets in a global variable

In [10]:
DATAFRAME_CACHE = {}

@tool
def preload_datasets(paths: List[str]) -> str:
    """
    Loads CSV files into a global cache if not already loaded.
    
    This function helps to efficiently manage datasets by loading them once
    and storing them in memory for future use. Without caching, you would
    waste tokens describing dataset contents repeatedly in agent responses.
    
    Args:
        paths: A list of file paths to CSV files.

    Returns:
        A message summarizing which datasets were loaded or already cached.
    """
    loaded = []
    cached = []
    for path in paths:
        if path not in DATAFRAME_CACHE:
            DATAFRAME_CACHE[path] = pd.read_csv(path)
            loaded.append(path)
        else:
            cached.append(path)
    
    return (
        f"Loaded datasets: {loaded}\n"
        f"Already cached: {cached}"
    )

### Summarization tool

In [20]:
from typing import List, Optional,Dict,Any

@tool
def get_dataset_summaries(dataset_paths: List[str]) -> List[Dict[str, Any]]:
    """
    Analyze multiple CSV files and return metadata summaries for each.

    Args:
        dataset_paths (List[str]): 
            A list of file paths to CSV datasets.

    Returns:
        List[Dict[str, Any]]: 
            A list of summaries, one per dataset, each containing:
            - "file_name": The path of the dataset file.
            - "column_names": A list of column names in the dataset.
            - "data_types": A dictionary mapping column names to their data types (as strings).
    """
    summaries = []

    for path in dataset_paths:
        # Load and cache the dataset if not already cached
        if path not in DATAFRAME_CACHE:
            DATAFRAME_CACHE[path] = pd.read_csv(path)
        
        df = DATAFRAME_CACHE[path]

        # Build summary
        summary = {
            "file_name": path,
            "column_names": df.columns.tolist(),
            "data_types": df.dtypes.astype(str).to_dict()
        }

        summaries.append(summary)

    return summaries

### DataFrame method execution tool

In [11]:
@tool
def call_dataframe_method(file_name: str, method: str) -> str:
   """
   Execute a method on a DataFrame and return the result.
   This tool lets you run simple DataFrame methods like 'head', 'tail', or 'describe' 
   on a dataset that has already been loaded and cached using 'preload_datasets'.
   Args:
       file_name (str): The path or name of the dataset in the global cache.
       method (str): The name of the method to call on the DataFrame. Only no-argument 
                     methods are supported (e.g., 'head', 'describe', 'info').
   Returns:
       str: The output of the method as a formatted string, or an error message if 
            the dataset is not found or the method is invalid.
   Example:
       call_dataframe_method(file_name="data.csv", method="head")
   """
   # Try to get the DataFrame from cache, or load it if not already cached
   if file_name not in DATAFRAME_CACHE:
       try:
           DATAFRAME_CACHE[file_name] = pd.read_csv(file_name)
       except FileNotFoundError:
           return f"DataFrame '{file_name}' not found in cache or on disk."
       except Exception as e:
           return f"Error loading '{file_name}': {str(e)}"
   
   df = DATAFRAME_CACHE[file_name]
   func = getattr(df, method, None)
   if not callable(func):
       return f"'{method}' is not a valid method of DataFrame."
   try:
       result = func()
       return str(result)
   except Exception as e:
       return f"Error calling '{method}' on '{file_name}': {str(e)}"

`getattr() function` allows us to retrieve and call a method using its string name.

This approach gives your agent the flexibility to select the most appropriate DataFrame method based on the current analysis needs.

In [14]:
my_string = "Hello World"
method_name = "upper" # The name of the method we want to call, as a string

upper_method = getattr(my_string, method_name) # Get the method using getattr()
result = upper_method() # call method using '()'
print(result)

HELLO WORLD


In [13]:
class Person:
    name = "John"
    age = 36
    country = "Norway"

person_object = Person()

# Standard attribute access
print(person_object.age)

# Using getattr()
print(getattr(person_object, 'age')) # Output: 36
print(getattr(person_object, 'name')) # Output: John


36
36
John


### Model evaluation tools

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, r2_score, mean_squared_error

# Assumes this global cache is shared
DATAFRAME_CACHE = {}

@tool
def evaluate_classification_dataset(file_name: str, target_column: str) -> Dict[str, float]:
    """
    Train and evaluate a classifier on a dataset using the specified target column.
    Args:
        file_name (str): The name or path of the dataset stored in DATAFRAME_CACHE.
        target_column (str): The name of the column to use as the classification target.
    Returns:
        Dict[str, float]: A dictionary with the model's accuracy score.
    """
    # Try to get the DataFrame from cache, or load it if not already cached
    if file_name not in DATAFRAME_CACHE:
        try:
            DATAFRAME_CACHE[file_name] = pd.read_csv(file_name)
        except FileNotFoundError:
            return {"error": f"DataFrame '{file_name}' not found in cache or on disk."}
        except Exception as e:
            return {"error": f"Error loading '{file_name}': {str(e)}"}
    
    df = DATAFRAME_CACHE[file_name]
    if target_column not in df.columns:
        return {"error": f"Target column '{target_column}' not found in '{file_name}'."}
    
    X = df.drop(columns=[target_column])
    y = df[target_column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = RandomForestClassifier()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return {"accuracy": acc}

@tool
def evaluate_regression_dataset(file_name: str, target_column: str) -> Dict[str, float]:
    """
    Train and evaluate a regression model on a dataset using the specified target column.
    Args:
        file_name (str): The name or path of the dataset stored in DATAFRAME_CACHE.
        target_column (str): The name of the column to use as the regression target.
    Returns:
        Dict[str, float]: A dictionary with R² score and Mean Squared Error.
    """
    # Try to get the DataFrame from cache, or load it if not already cached
    if file_name not in DATAFRAME_CACHE:
        try:
            DATAFRAME_CACHE[file_name] = pd.read_csv(file_name)
        except FileNotFoundError:
            return {"error": f"DataFrame '{file_name}' not found in cache or on disk."}
        except Exception as e:
            return {"error": f"Error loading '{file_name}': {str(e)}"}
    
    df = DATAFRAME_CACHE[file_name]
    if target_column not in df.columns:
        return {"error": f"Target column '{target_column}' not found in '{file_name}'."}
    
    X = df.drop(columns=[target_column])
    y = df[target_column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = RandomForestRegressor()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    return {
        "r2_score": r2,
        "mean_squared_error": mse
    }

## Create Agent

### Agent without ReAct pattern

This agent will just call one tool and will return response. It will not break problem and will not call multiple tools step-by-step.\
`It will return response of 1st tool_call.`

`create_openai_tools_agent` agent has significant limitations when used directly. It only performs a single step of reasoning and tool usage per invocation, then returns its intermediate thought process rather than a final answer. This behavior occurs because the agent doesn't automatically manage the full loop of thinking, acting, observing results, and continuing to reason until reaching a complete solution.

In [18]:
prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "You are a data science assistant. Use the available tools to analyze CSV files. "
     "Your job is to determine whether each dataset is for classification or regression, based on its structure."),
    
    ("user", "{input}"),
    ("placeholder", "{agent_scratchpad}")  # Required for tool-calling agents
])

llm = init_chat_model("gpt-4o-mini", model_provider="openai", streaming=False )

/tmp/ipykernel_3400265/714181554.py:16: LangChainBetaWarning: The function `init_chat_model` is in beta. It is actively being worked on, so the API may change.
  llm = init_chat_model("gpt-4o-mini", model_provider="openai", streaming=False )


In [21]:
tools=[list_csv_files, preload_datasets, get_dataset_summaries, call_dataframe_method, evaluate_classification_dataset, evaluate_regression_dataset]

In [23]:
# Construct the tool calling agent
agent = create_openai_tools_agent(llm, tools, prompt)

In [24]:
response = agent.invoke({
    "input": "Can you tell me about the dataset?",
    "intermediate_steps": []
})

In [26]:
response

[ToolAgentAction(tool='list_csv_files', tool_input={}, log='\nInvoking: `list_csv_files` with `{}`\n\n\n', message_log=[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_oE0KWeHlFDA1EbAxtgpVbmrU', 'function': {'arguments': '{}', 'name': 'list_csv_files'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 669, 'total_tokens': 680, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f4ae844694', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-d1e30bf0-1086-4cbd-8dfd-c7a32c1b2ccd-0', tool_calls=[{'name': 'list_csv_files', 'args': {}, 'id': 'call_oE0KWeHlFDA1EbAxtgpVbmrU', 'type': 'tool_call'}], usage_metadata={'input_tokens': 669, 'output_tokens': 11, 'total_tokens': 680})], tool_call_id='call_o

In [25]:
# Get the first ToolAgentAction from the list
action = response[0]

# Print the key details
print("🧠 Agent decided to call a tool:")
print("Tool Name:", action.tool)
print("Tool Input:", action.tool_input)
print("Log:\n", action.log.strip())

🧠 Agent decided to call a tool:
Tool Name: list_csv_files
Tool Input: {}
Log:
 Invoking: `list_csv_files` with `{}`


### Agent with ReAct pattern

`ReAct-style agents` follow a step-by-step reasoning loop. ReAct stands for Reasoning and Acting: the agent thinks about what to do next, takes one action (like calling a tool), then waits for the result before continuing.

`AgentExecutor` wraps the agent and the toolset, and handles the full tool-use loop behind the scenes. It automatically runs the agent, executes the selected tool, takes the result (observation), and feeds it back into the agent until a final answer is reached.\
`Without the executor`, you'd have to manually manage every step, including checking whether the agent returned a tool call or a final answer, running the tool, and tracking the intermediate steps—all of which the executor handles for you.

In [30]:
agent_executor = AgentExecutor(agent=agent, 
                               tools=tools, 
                               verbose=True, # Enables detailed logging of each step in the agent's thinking and tool-calling process.
                               handle_parsing_errors=True # Rather than crashing, the executor will attempt to recover and continue the conversation.
                               )
agent_executor.agent.stream_runnable = False # disables streaming mode for the agent.



**NOTE**: Instead of `AgentExecutor` use create_agent from langchain>=V1.0. Check 1.agents-and-tools.ipynb for its implementation

In [32]:
print("📊 Ask questions about your dataset (type 'exit' to quit):")

while True:
    user_input=input(" You:")
    if user_input.strip().lower() in ['exit','quit']:
        print("see ya later")
        break
        
    result=agent_executor.invoke({"input":user_input})
    print(f"my Agent: {result['output']}")

📊 Ask questions about your dataset (type 'exit' to quit):


> Entering new AgentExecutor chain...

Invoking: `list_csv_files` with `{}`


['regression-dataset.csv', 'classification-dataset.csv']
Invoking: `preload_datasets` with `{'paths': ['classification-dataset.csv']}`


Loaded datasets: []
Already cached: ['classification-dataset.csv']
Invoking: `get_dataset_summaries` with `{'dataset_paths': ['classification-dataset.csv']}`


[{'file_name': 'classification-dataset.csv', 'column_names': ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'mean compactness', 'mean concavity', 'mean concave points', 'mean symmetry', 'mean fractal dimension', 'radius error', 'texture error', 'perimeter error', 'area error', 'smoothness error', 'compactness error', 'concavity error', 'concave points error', 'symmetry error', 'fractal dimension error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst smoothness', 'worst compactness', 'worst concavity

`QUESTION FOR ABOVE^ WAS: List number of columns in classification dataset`

In [33]:
result=agent_executor.invoke({"input": "Find important columns for classification dataset and then perform classification for target column"})
print(f"my Agent: {result['output']}")



> Entering new AgentExecutor chain...

Invoking: `list_csv_files` with `{}`


['regression-dataset.csv', 'classification-dataset.csv']
Invoking: `preload_datasets` with `{'paths': ['classification-dataset.csv']}`


Loaded datasets: []
Already cached: ['classification-dataset.csv']
Invoking: `get_dataset_summaries` with `{'dataset_paths': ['classification-dataset.csv']}`


[{'file_name': 'classification-dataset.csv', 'column_names': ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'mean compactness', 'mean concavity', 'mean concave points', 'mean symmetry', 'mean fractal dimension', 'radius error', 'texture error', 'perimeter error', 'area error', 'smoothness error', 'compactness error', 'concavity error', 'concave points error', 'symmetry error', 'fractal dimension error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst smoothness', 'worst compactness', 'worst concavity', 'worst concave points', 'worst symmetry', 'worst fracta

In [34]:
result

{'input': 'Find important columns for classification dataset and then perform classification for target column',
 'output': 'The classification dataset has been analyzed, and the classification model achieved an accuracy of approximately **95.61%** when predicting the target column. \n\nIf you need further analysis or actions, feel free to ask!'}